# Handicapper Wisdom of Crowds – Notebook Interface

GitHub-Native CSV workflow. No Streamlit / Vercel required.

**Data directory:** `data/handicapper/`

In [ ]:
# Cell 1: Setup & Load Data
from data_loader import CSVDataManager
from parser import HandicapperParser
from team_normalizer import CBBNormalizer
from game_mapper import GameMapper
import pandas as pd

dm = CSVDataManager()
data = dm.load_app_data()

print('📁 Data loaded:')
for name, df in data.items():
    print(f'  {name}: {len(df)} rows')

In [ ]:
# Cell 2: Add Handicapper (Notebook Style)
new_capper = {
    'handle': '@NewSharpGuy',
    'tier': 'sharp',
    'status': 'active',
    'lifetime_roi': 0.0,
    'win_pct': 0.5,
    'total_picks': 0,
    'parse_template_id': 1,
    'notes': 'Testing new capper',
    'created_at': pd.Timestamp.now().isoformat()
}

capper_id = dm.append_record('handicappers', new_capper)
print(f'✅ Added handicapper ID: {capper_id}')

# Reload and show the newly added row
data = dm.load_app_data()
data['handicappers'].tail(1)

In [ ]:
# Cell 3: Parse Tweet Example
parser = HandicapperParser(dm.data_dir)

tweet = {
    'text': 'Illinois -3.5 (2u) | Purdue ML | UConn/Kentucky O142.5',
    'handicapper_id': 1,
    'tweet_id': '1849999999999999999'
}

raw_picks = parser.parse_tweet_to_raw_picks(**tweet)
parser.save_raw_picks(raw_picks)

print('Parsed picks:')
for pick in raw_picks:
    print(f'  {pick}')

In [ ]:
# Cell 4: Map Picks to Games
mapper = GameMapper(dm.data_dir)

# Reload so we see any newly parsed picks
data = dm.load_app_data()
raw_picks_df = data['raw_picks'][data['raw_picks']['parse_status'] == 'success']

mapping_results = mapper.batch_map_raw_picks(raw_picks_df, data['games'])
mapper.save_picks(mapping_results)

print('✅ Mapping complete:')
display_cols = [c for c in ['raw_pick_id', 'game_id', 'mapping_status'] if c in mapping_results.columns]
print(mapping_results[display_cols].head())

In [ ]:
# Cell 5: Review Final State
data = dm.load_app_data()
print('📊 Final data state:')
for name, df in data.items():
    print(f'  {name}: {len(df)} rows')

print('\n🏀 Picks summary:')
if not data['picks'].empty:
    print(data['picks'][['pick_id', 'handicapper_id', 'game_id', 'market', 'mapping_status']].to_string(index=False))